In [1]:
#Creating the test dataset
test_cases = [
    {
        "id": "rag_001",
        "question": "What is RAG?",
        "expected_answer": (
            "RAG retrieves relevant information from documents "
            "and provides it to a language model to generate an answer."
        ),
        "expected_keywords": [
            "retrieval",
            "documents",
            "generate"
        ]
    },
    {
        "id": "rag_002",
        "question": "What are embeddings?",
        "expected_answer": (
            "Embeddings convert text into numerical vectors "
            "that represent semantic meaning."
        ),
        "expected_keywords": [
            "text",
            "numerical vectors",
            "meaning"
        ]
    },
    {
        "id": "rag_003",
        "question": "What does a vector database do?",
        "expected_answer": (
            "A vector database stores embeddings and supports "
            "similarity searches."
        ),
        "expected_keywords": [
            "stores",
            "embeddings",
            "similarity"
        ]
    },
    {
        "id": "rag_004",
        "question": "Why do we chunk documents?",
        "expected_answer": (
            "Chunking divides long documents into smaller pieces "
            "so relevant information can be retrieved efficiently."
        ),
        "expected_keywords": [
            "smaller",
            "chunks",
            "retrieval"
        ]
    },
    {
        "id": "rag_005",
        "question": "What is semantic search?",
        "expected_answer": (
            "Semantic search finds information based on meaning "
            "rather than only exact keyword matches."
        ),
        "expected_keywords": [
            "meaning",
            "information",
            "keywords"
        ]
    }
]

In [2]:
#Saving the datset as json
import json

with open("eval_dataset.json", "w", encoding="utf-8") as file:
    json.dump(test_cases, file, indent=4)

print("Evaluation dataset saved!")

Evaluation dataset saved!


In [4]:
#Creating the Keyword coverage function
import re

def normalize_text(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    return " ".join(words)


def calculate_keyword_coverage(answer, expected_keywords):

    normalized_answer = normalize_text(answer)

    matched_keywords = []

    for keyword in expected_keywords:

        normalized_keyword = normalize_text(keyword)

        searchable_answer = f" {normalized_answer} "
        searchable_keyword = f" {normalized_keyword} "

        if searchable_keyword in searchable_answer:
            matched_keywords.append(keyword)

    if not expected_keywords:
        return 0.0, matched_keywords

    score = len(matched_keywords) / len(expected_keywords)

    return score, matched_keywords

In [5]:
#Testing the metric
sample_answer = (
    "RAG uses retrieval to find relevant documents "
    "and generate a response."
)

score, matched = calculate_keyword_coverage(
    sample_answer,
    ["retrieval", "documents", "generate"]
)

print("Score:", score)
print("Matched keywords:", matched)

Score: 1.0
Matched keywords: ['retrieval', 'documents', 'generate']


## Building the Evaluation Harness

In [6]:
#Creating a RAG Answer function
demo_answers = {
    "What is RAG?": (
        "RAG uses retrieval to find relevant documents "
        "and generate a response."
    ),

    "What are embeddings?": (
        "Embeddings convert text into numerical vectors "
        "that represent meaning."
    ),

    "What does a vector database do?": (
        "A vector database stores embeddings "
        "and supports similarity search."
    ),

    "Why do we chunk documents?": (
        "Chunking splits a long document into pieces."
    ),

    "What is semantic search?": (
        "Semantic search finds information based on meaning "
        "rather than only exact keywords."
    )
}


def demo_rag_answer(question):

    return demo_answers.get(
        question,
        "I do not have an answer for this question."
    )

In [8]:
#Evaluationg every test case
import time


def run_evaluation(
    answer_function,
    test_cases,
    pass_threshold=0.65
):

    results = []

    for case in test_cases:

        start_time = time.perf_counter()

        try:

            raw_answer = answer_function(case["question"])

            if isinstance(raw_answer, dict):
                answer = raw_answer.get("answer", "")
            else:
                answer = str(raw_answer)

            score, matched = calculate_keyword_coverage(
                answer,
                case["expected_keywords"]
            )

            passed = score >= pass_threshold
            error = ""

        except Exception as exc:

            answer = ""
            score = 0.0
            matched = []
            passed = False
            error = str(exc)

        latency = time.perf_counter() - start_time

        results.append({
            "id": case["id"],
            "question": case["question"],
            "answer": answer,
            "keyword_coverage": round(score, 3),
            "matched_keywords": ", ".join(matched),
            "passed": passed,
            "latency_seconds": round(latency, 4),
            "error": error
        })

    return results

In [9]:
#Running the Evaluation
results = run_evaluation(
    demo_rag_answer,
    test_cases
)

for result in results:

    print("Question:", result["question"])
    print("Answer:", result["answer"])
    print("Score:", result["keyword_coverage"])
    print("Passed:", result["passed"])
    print("Latency:", result["latency_seconds"])
    print("-" * 50)

Question: What is RAG?
Answer: RAG uses retrieval to find relevant documents and generate a response.
Score: 1.0
Passed: True
Latency: 0.0001
--------------------------------------------------
Question: What are embeddings?
Answer: Embeddings convert text into numerical vectors that represent meaning.
Score: 1.0
Passed: True
Latency: 0.0
--------------------------------------------------
Question: What does a vector database do?
Answer: A vector database stores embeddings and supports similarity search.
Score: 1.0
Passed: True
Latency: 0.0
--------------------------------------------------
Question: Why do we chunk documents?
Answer: Chunking splits a long document into pieces.
Score: 0.0
Passed: False
Latency: 0.0
--------------------------------------------------
Question: What is semantic search?
Answer: Semantic search finds information based on meaning rather than only exact keywords.
Score: 1.0
Passed: True
Latency: 0.0
--------------------------------------------------


In [10]:
#Creating the Summary function
def summarize_results(results):

    total_tests = len(results)

    if total_tests == 0:
        print("No tests were executed.")
        return

    passed_tests = sum(
        1 for result in results
        if result["passed"]
    )

    pass_rate = passed_tests / total_tests

    average_coverage = sum(
        result["keyword_coverage"]
        for result in results
    ) / total_tests

    average_latency = sum(
        result["latency_seconds"]
        for result in results
    ) / total_tests

    print("Total tests:", total_tests)
    print("Passed tests:", passed_tests)
    print("Failed tests:", total_tests - passed_tests)
    print("Pass rate:", round(pass_rate * 100, 2), "%")
    print("Average keyword coverage:", round(average_coverage, 3))
    print("Average latency:", round(average_latency, 4), "seconds")

In [11]:
summarize_results(results)

Total tests: 5
Passed tests: 4
Failed tests: 1
Pass rate: 80.0 %
Average keyword coverage: 0.8
Average latency: 0.0 seconds


In [12]:
#Save the evaluation results
import csv

fieldnames = [
    "id",
    "question",
    "answer",
    "keyword_coverage",
    "matched_keywords",
    "passed",
    "latency_seconds",
    "error"
]

with open(
    "evaluation_results.csv",
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=fieldnames
    )

    writer.writeheader()
    writer.writerows(results)

print("Evaluation results saved!")

Evaluation results saved!


In [13]:
def answer_with_day21_rag(question):

    result = chat_with_documents(question)

    if isinstance(result, dict):
        return result.get("answer", "")

    return result

In [14]:
real_results = run_evaluation(
    answer_with_day21_rag,
    test_cases
)

summarize_results(real_results)

Total tests: 5
Passed tests: 0
Failed tests: 5
Pass rate: 0.0 %
Average keyword coverage: 0.0
Average latency: 0.0 seconds
